In [1]:
from orion.algos.human_video_oogs import HumanVideoOOGs
from orion.algos.oog import OpenWorldObjectSceneGraph
from orion.utils.misc_utils import *
from orion.utils.o3d_utils import *
from orion.utils.correspondence_utils import CorrespondenceModel
human_video_oogs = HumanVideoOOGs()
# correspondence_model = CorrespondenceModel()
# human_video_oogs.set_correspondence_model(correspondence_model)

[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /data/arisrei/miniconda3/envs/ORION/lib/python3.9/site-packages/robosuite/scripts/setup_macros.py (macros.py:55)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
human_video_annotation_path = "annotations/human_demo/iphone_front_boat_demo"
human_video_oogs.init()
human_video_oogs.generate_from_human_video(human_video_annotation_path)


temporal_segments:  TemporalSegments(name=, num_segments=3, seq_len=164)
Plane equation: -0.01x + 0.87y + 0.50z + -0.35 = 0
Number of inliers: 98785
2.0956201937431977
Estimated transformation matrix is: 
[[ 0.99992376  0.01069769 -0.00616774  0.        ]
 [ 0.01069769 -0.50098434  0.86539023  0.        ]
 [ 0.00616774 -0.86539023 -0.50106058  0.        ]
 [ 0.          0.          0.          1.        ]]
Pixel traj shape:  (80, 38, 2)
Pixel traj shape:  (80, 46, 2)
Pixel traj shape:  (80, 76, 2)
Pixel traj shape:  (80, 1, 2)
Generated # of human_object_graphs:  4


In [3]:
idx = 1

human_video_oogs.get_graph(0).draw_overlay_image(mode="object")

input_annotation = human_video_oogs.get_graph(idx).input_annotation
overlay_image = overlay_xmem_mask_on_image(
                human_video_oogs.get_graph(idx).input_image, 
                human_video_oogs.get_graph(idx).input_annotation,
                use_white_bg=False,
                rgb_alpha=0.3)



overlay_image = overlay_image * (input_annotation == 0)[..., None] + human_video_oogs.get_graph(idx).input_image * (input_annotation != 0)[..., None]
plotly_draw_image(overlay_image, width=640, height=480)


Plotting object overlay


### Source image and depth

In [4]:
plotly_draw_image(human_video_oogs.get_graph(1).input_image, width=640, height=480)
plotly_draw_image(human_video_oogs.get_graph(0).input_depth, width=640, height=480)
human_video_oogs.get_graph(0).draw_overlay_image(mode="point", width=640, height=480, default_point_color="rgb(30, 122, 122)")

Plotting point overlay


### Draw the scene and the objects from human video
1. Draw the scene
2. Draw the scene with objects being annotated with segmentation mask
3. Trajectories that show the 3D motion of the objects

In [5]:
human_video_oogs.get_graph(1).draw_scene_3d(downsample=True, 
                                            depth_trunc=1.0,  
                                            use_white_bg=True,
                                            no_background=True, 
                                            overlay_annotation=True)

(array([[-0.23385741,  0.6781661 , -0.00933881],
        [-0.23034919,  0.67705581, -0.00930386],
        [-0.22511415,  0.68973801, -0.00941804],
        ...,
        [ 0.12601261,  0.6781552 , -0.68835653],
        [ 0.185639  ,  0.72677298, -0.73684247],
        [ 0.52069789,  0.72236098, -0.72663348]]),
 array([[0.62745098, 0.62745098, 0.62745098],
        [0.63137255, 0.63137255, 0.63137255],
        [0.63137255, 0.63137255, 0.63137255],
        ...,
        [0.65098039, 0.65098039, 0.65098039],
        [0.63921569, 0.63921569, 0.63921569],
        [0.68235294, 0.67843137, 0.66666667]]))

In [6]:
trajs = human_video_oogs.get_graph(1).get_world_trajs(object_ids=[2])
trajs = np.delete(trajs, 34, axis=0)
human_video_oogs.get_graph(1).draw_objects_list_3d(object_ids=[1,2], downsample=True, no_background=True)

(array([[ 0.15280545,  0.74268388, -0.24194898],
        [ 0.15434524,  0.74367158, -0.24225782],
        [ 0.16147304,  0.75831628, -0.24698891],
        ...,
        [-0.05485526,  0.77018792, -0.35354716],
        [-0.07858701,  0.79663549, -0.29360669],
        [-0.07323147,  0.7808426 , -0.30298687]]),
 array([[0.21568627, 0.21568627, 0.17647059],
        [0.22352941, 0.22352941, 0.18431373],
        [0.21176471, 0.21176471, 0.17254902],
        ...,
        [0.5254902 , 0.19215686, 0.21960784],
        [0.59607843, 0.49803922, 0.47843137],
        [0.58039216, 0.24705882, 0.22745098]]))

In [7]:
trajs = human_video_oogs.get_graph(1).get_world_trajs(object_ids=[2])
trajs = np.delete(trajs, [10, 17], axis=0)
human_video_oogs.get_graph(0).draw_scene_3d(
                                            additional_points=trajs,
                                            downsample=True, no_background=True,
                                            depth_trunc=1.0,  overlay_annotation=True, 
                                            additional_point_draw_lines=True,)

(38, 46, 3)


(array([[-0.2062053 ,  0.51849244, -0.07190724],
        [-0.20702532,  0.51335107, -0.07279383],
        [-0.19871358,  0.52691038, -0.07542261],
        ...,
        [ 0.15167192,  0.64862403, -0.65784972],
        [ 0.16323699,  0.6640141 , -0.67332277],
        [ 0.53832273,  0.72763829, -0.7317062 ]]),
 array([[0.11372549, 0.09411765, 0.07058824],
        [0.18823529, 0.15686275, 0.1372549 ],
        [0.17647059, 0.14509804, 0.10588235],
        ...,
        [0.03137255, 0.03137255, 0.02352941],
        [0.03137255, 0.03137255, 0.02352941],
        [0.0627451 , 0.05098039, 0.03529412]]))

### Draw individual object point clouds

In [ ]:
human_video_oogs.get_graph(1).draw_objects_3d(object_id=2, downsample=False,  no_background=True, draw_keypoints=True, keypoints_rgb_str="(30, 200, 200)", uniform_color=True)

In [ ]:
trajs = human_video_oogs.get_graph(1).get_world_trajs(object_ids=[1])
trajs = np.delete(trajs, [10, 17], axis=0)
human_video_oogs.get_graph(1).draw_objects_3d(object_id=1, downsample=False,  no_background=True, draw_trajs=False, additional_point_draw_lines=True, keypoints_rgb_str="(30, 122, 122)", 
                                              additional_points=trajs, 
                                              uniform_color=False)

In [ ]:
import trimesh
import plotly.graph_objects as go

# Load the .obj file using trimesh
mesh = trimesh.load('annotations/human_demo/iphone_front_boat_demo/hamer_output/0039_0.obj')

# Extract vertices and faces
vertices = mesh.vertices
faces = mesh.faces

# Create a 3D plot
fig = go.Figure(data=[go.Mesh3d(x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2], 
                                i=faces[:, 0], j=faces[:, 1], k=faces[:, 2], 
                                 opacity=0.50)])

# Adjust the camera view as desired
fig.update_layout(scene_camera=dict(
    eye=dict(x=1.5, y=1.5, z=1.5)  # Example camera position
),
        scene=dict(
            xaxis=dict(showbackground=False, zeroline=False, showgrid=False, showticklabels=False, showaxeslabels=False, visible=False),
            yaxis=dict(showbackground=False, zeroline=False, showgrid=False, showticklabels=False, showaxeslabels=False, visible=False),
            zaxis=dict(showbackground=False, zeroline=False, showgrid=False, showticklabels=False, showaxeslabels=False, visible=False),
        ),
        paper_bgcolor='rgba(0,0,0,0)',  # Transparent background
        plot_bgcolor='rgba(0,0,0,0)',  # Transparent background
        margin=dict(l=0, r=0, b=0, t=0),  # No margins
        showlegend=False,
)

fig.show()